In [ ]:
from pathlib import Path

REPO = "/content/ProtoFormer"
GIT_URL = "https://github.com/pratamabintang/PF.git"
BRANCH = "main"

DATA = "/content/data"
SAM2_CKPT = "/content/sam2_hiera_l.pth"

WEIGHT_1SHOT = "/content/drive/MyDrive/ProtoFormer_out/best_model_1shot.pt"
WEIGHT_5SHOT = "/content/drive/MyDrive/ProtoFormer_out/best_model_5shot.pt"

BENCHMARK = "pascal"
FOLD = 0
BSZ = 1
IMG_SIZE = 512
REDUCE_DIM = 64
AMP = "auto"

print(locals())

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
if not Path(REPO).exists():
    !git clone -b {BRANCH} {GIT_URL} {REPO}
else:
    !cd {REPO} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

print(Path(REPO, "test.py").exists())

In [ ]:
!cd {REPO} && pip install -q -r requirements.txt

In [ ]:
import sys

sys.path.insert(0, str(Path(REPO, "model")))
sys.path.insert(0, REPO)

import sam2
print("sam2:", sam2.__file__)

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
checks = {
    "repo/test.py": Path(REPO, "test.py"),
    "repo/model/sam2": Path(REPO, "model/sam2"),
    "data/VOC2012/JPEGImages": Path(DATA, "VOC2012/JPEGImages"),
    "data/VOC2012/SegmentationClassAug": Path(DATA, "VOC2012/SegmentationClassAug"),
    "sam2 checkpoint": Path(SAM2_CKPT),
    "weight 1-shot": Path(WEIGHT_1SHOT),
    "weight 5-shot": Path(WEIGHT_5SHOT),
}

for name, p in checks.items():
    print(f"{name:38s} {p.exists()!s:6s} {p}")

In [ ]:
!cd {REPO} && PYTHONPATH=model python test.py --datapath {DATA} --benchmark {BENCHMARK} --fold {FOLD} --shot 1 --bsz {BSZ} --reduce_dim {REDUCE_DIM} --img_size {IMG_SIZE} --amp {AMP} --sam2_checkpoint {SAM2_CKPT} --load {WEIGHT_1SHOT}

In [ ]:
!cd {REPO} && PYTHONPATH=model python test.py --datapath {DATA} --benchmark {BENCHMARK} --fold {FOLD} --shot 5 --bsz {BSZ} --reduce_dim {REDUCE_DIM} --img_size {IMG_SIZE} --amp {AMP} --sam2_checkpoint {SAM2_CKPT} --load {WEIGHT_5SHOT}

In [ ]:
import glob

for f in sorted(glob.glob(f"{REPO}/logs/*/log.txt")):
    print("==", f)
    print(Path(f).read_text()[-1500:])